In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

In [2]:
PROJECT_ROOT = "/Users/ashritkuma.samudrala/lnex/ex_llm_rag"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
save_path = f"{PROJECT_ROOT}/main/resources/models/char_model/gpt_char_model3_5000iters.model"
checkpoint = torch.load(save_path, map_location=device)

In [3]:
# Restore hyperparams first
hp = checkpoint['hyperparams']
context_len = hp['context_len']
emb_dim = hp['emb_dim']
n_heads = hp['n_heads']
n_transformer_blocks = hp['n_transformer_blocks']
dropout = hp['dropout']
vocab_size = 65

In [ ]:
# Lets add resudial connections and layer norm

class AttnHead_N(nn.Module):
    def __init__(self, attn_head_size):
        super().__init__()
        self.attn_head_size = attn_head_size
        self.Qw = nn.Linear(emb_dim, attn_head_size)
        self.Kw = nn.Linear(emb_dim, attn_head_size)
        self.Vw = nn.Linear(emb_dim, attn_head_size)
        self.register_buffer('tril', torch.tril(torch.ones(context_len, context_len)))

        # This will randomly swuth off some neurons (dropout%) during training to prevent overfitting
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x): # X (B, CL, ED)
        B, CL, ED = x.shape
        Q = self.Qw(x) # (B, CL, attn_head_size)
        K = self.Kw(x) # (B, CL, attn_head_size)
        V = self.Vw(x) # (B, CL, attn_head_size)
        
        w = Q @ K.transpose(1, 2) # (B, CL, attn_head_size) @ (B, attn_head_size, CL) = (B, CL, CL)
        w = w * (self.attn_head_size ** -0.5)
        # We need to do :CL, :CL instead of sequence_len, since in generation we might have less than context_len size
        w = w.masked_fill(self.tril[:CL, :CL] == 0, float('-inf'))
        w = F.softmax(w, dim=-1)

        # we drop out before finally getting the attention o/p so that we can switch off few neurons from comunicating (prevent overfitting)
        w = self.dropout(w)
        
        # Perform weighted aggregation of values
        out = w @ V # (B, CL, CL) @ (B, CL, attn_head_size) = (B, CL, attn_head_size)
        return out


# It helps to have multiple channels of communicatin (heads) per attention block so that each of the head can commnicate separately and gather different kinds of data and finally they are all mixed together
class MultiHeadAttn_N(nn.Module):
    def __init__(self, n_heads):
        super().__init__()
        self.heads = nn.ModuleList([AttnHead_N(emb_dim//n_heads) for _ in range(n_heads)]) # creates 4 4 self attn heads each 8 dimensional

        # This also called W0.
        # This will linearly mixes (transform) the attention output
        self.proj = nn.Linear(emb_dim, emb_dim)

        # This will randomly swuth off some neurons (dropout%) during training to prevent overfitting
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x): # (B, CL, ED)
        out = torch.cat([head(x) for head in self.heads], dim=-1) # head(x) would return (B, CL, attn_head_size), concatinating all of them over last dim attn_head_size would give us back (B, CL, ED)
        out = self.proj(out) #(B, CL, ED)
        out = self.dropout(out)
        return out


class FeedForwardNetwork_N(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.ffn(x)

class TransformerBlock_N(nn.Module):
    def __init__(self, n_heads, emb_dim):
        super().__init__()
        self.m_head_attn = MultiHeadAttn_N(n_heads)
        self.ffn = FeedForwardNetwork_N(emb_dim)
        # layer norm before attention
        self.ln1 = nn.LayerNorm(emb_dim)

        # layer norm before mlp
        self.ln2 = nn.LayerNorm(emb_dim)
    
    def forward(self, x):
        x = x + self.m_head_attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class GPTModel3(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim) # (V, ED)
        self.positions = nn.Embedding(context_len, emb_dim) # (CL, ED)
        
        self.blocks = nn.Sequential(
            *[TransformerBlock_N(n_heads, emb_dim) for _ in range(n_transformer_blocks)] # '*' like the spread (...) operator in JS, spreads the blocks in the list
        )

        # Final layer normalization
        self.ln_f = nn.LayerNorm(emb_dim)

        self.lm_head = nn.Linear(emb_dim, vocab_size) # (ED, V)
    
    def forward(self, x, targets=None): # x => (B, CL), targets => (B, CL)
        B, CL = x.shape
        tok_emb = self.embedding(x) # (B, CL, ED)
        pos_emb = self.positions(torch.arange(CL, device=device)) # (CL, ED), torch.arange create 0 to CL-1 ints, we pluck those indexes out of positions
        # print(f"Tok emb = {tok_emb.shape}, pos emb = {pos_emb.shape}")
        x = tok_emb + pos_emb # (B, CL, ED)
        x = self.blocks(x) # (B, CL, ED)
        x = self.ln_f(x) # (B, CL, ED)
        logits = self.lm_head(x) # (B, CL, V)
        if targets is not None:
            B, CL, V = logits.shape
            # cross entropy expects either (B*CL, V) or (B, V, CL) so we need to convert
            logits = logits.view(B*CL, V)
            targets = targets.view(B*CL)
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None
        return logits, loss
    
    def generate(self, inp, max_len): # inp (B, CL)
        for _ in range(max_len):
            # Crop till context length
            inp_ctx = inp[:, -context_len:]
            logits, _ = self.forward(inp_ctx) # (B, CL, V)
            # We only need the last token logits in every batch
            logits = logits[:, -1, :] # (B, V)

            # Apply SFMX along the last dim, i.e last token raw logits in every batch
            probs = F.softmax(logits, dim=-1) # (B, V)

            # sample, this gives next token index for every batch
            next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
            inp = torch.cat([inp, next_token], dim=1) # (B, CL+1)
        
        return inp

# gpt_model3 = GPTModel3()
# gpt_model3.to(device)

GPTModel3(
  (embedding): Embedding(65, 384)
  (positions): Embedding(256, 384)
  (blocks): Sequential(
    (0): TransformerBlock_N(
      (m_head_attn): MultiHeadAttn_N(
        (heads): ModuleList(
          (0-5): 6 x AttnHead_N(
            (Qw): Linear(in_features=384, out_features=64, bias=True)
            (Kw): Linear(in_features=384, out_features=64, bias=True)
            (Vw): Linear(in_features=384, out_features=64, bias=True)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffn): FeedForwardNetwork_N(
        (ffn): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
    

In [5]:
# Rebuild model and load weights
loaded_model = GPTModel3()
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.to(device)
loaded_model.eval()

print(f"Loaded model — train loss: {checkpoint['train_loss']}, val loss: {checkpoint['val_loss']}")

Loaded model — train loss: 1.0896, val loss: 1.4894


In [6]:
vocab = ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

c2i = {ch: i for i, ch in enumerate(vocab)}
i2c = {i: ch for i, ch in enumerate(vocab)}

def encode(test: str):
    return [c2i[ch] for ch in test]

def decode(tokens: list[int]):
    return ''.join([i2c[i] for i in tokens])

In [16]:
i = encode("so happy my friend")
inp = torch.tensor([i], dtype=torch.long, device=device)
print(inp.shape)
op_tox = loaded_model.generate(inp, 200)
# print(op_tox.shape)
print(decode(op_tox[0].tolist()))

torch.Size([1, 18])
so happy my friend,
By noble from me slain against their royal unagon.

NORTHUMBERLAND:
Yes, I must be barren myself:
Adoubt thy ancient passage, am thy son.

KING RICHARD III:
O, but, not hear me.

KING RICHARD II:
Th
